# Prompting

Prompting is the most done Machine Learning task ever. Every single time a user uses: ChatGpt, the browser in google chrome, the chat bot for claude, anything, you are prompting a language model to do a task. 

## Types or prompts

There are three types of prompts for these language models they are as follows.

### System Prompts, User Prompts, Assistant Prompts (Roles)

The **system** prompt tells the model how to behave. All chat bots have this. Usually the system prompt is not added right after pre-training. So, think of it like this: the system prompt tells the model how to behave. ALL MODELS THAT ARE COMMERCIAL HAVE THIS. There is something call `prompt-injection`. This is when you get the model (ChatGPT, Gemini, Claude ect.) to reveal it's system prompt. Do not do that, but most of them have been leaked.

**User Prompt**: The user prompt is the input you give the `AI` model. These can be seen in a JSON file format if you ever download you chat history with a chat bot from most providers.

**NOTE**: If you use Lang Chain, it calls the `user prompt` `human` as the prompt role.. The `assistant` prompts are what the model generates back to you. This is the response you get from a prompt on ChatGPT or Claude ect. Lang Chain calls the `assistant` prompt the `ai`. It is the same thing however.

## Messages

Messages are what you give as input in any way to the language model.


## System, Assistant, and User Prompt Example

The system prompt, as mentioned, tells the model how to behave. This is example using Qwen (you cannot do this with base models.) I will give a example of the LLM talking like a pirate to the system prompt.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda:0"

# qwen from hugging face hub
model_name = "Qwen/Qwen2.5-7B-Instruct"


# the system prompt to tell the LLM how to behave.

SYSTEM_PROMPT = "You are a helpful assistant that answers briefly in the tone of pirate."


# messages with roles.
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "What is the president of the united states based on your training data?"}, # this is what the user tells the model.
    {"role": "assistant", "content": "the president of the united states is joe biden."}, # this is the response from the model.
    {"role": "user", "content": "And what state is he from?"},
]


# load in tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)

# load in model
model = AutoModelForCausalLM.from_pretrained(model_name)



# inputs to the llm
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=50, do_sample=False)

response = tokenizer.decode(
    output_ids[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)

print("\nResponse from LLM")
print()
(response)

/home/nick/github-projects/prompting-base-models/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 339/339 [00:00<00:00, 1295.10it/s]



Response from LLM



"Ah, he be comin' from the great state of Delaware!"

### About the output

That example was given the model messages based on the training data. That model does not have access to the current president of the United States. So, said Joe Biden, Even if it was up-to date, it would still have to obey the rules from the system prompt.

## Reasoning Models

Reasoning models became widely known in 2024 after OpenAI introduced
`o1-preview`. These models are designed to spend additional computation
working through complex problems before producing a final answer.

Some reasoning models display an intermediate reasoning process before their
final response:

![Example of a reasoning model response](/home/nick/github-projects/prompting-base-models/assets/images/reasoning-example.png)

Reasoning models have build in `<think>` tags. In between these tags includes Chain of thought prompting.[CoT-prompting-paper](https://arxiv.org/pdf/2201.11903). These models use more tokens because the output consisting of the model reasoning about the response before it applies

This is example is from the following: [link](https://huggingface.co/Qwen/Qwen3-0.6B)

## Example of Reasoning Model in Python

In [2]:
model_name = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, dtype="auto", device_map="auto")

messages = [{"role": "user", "content": "What is 17 * 24?"}]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    enable_thinking=True,
    return_tensors="pt",
).to(model.device)

output_ids = model.generate(
    **inputs,
    max_new_tokens=2048,
    do_sample=True,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
)

response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
thinking, answer = response.split("</think>")

print("==== Thinking ====")
print(thinking.strip())
print()
print("==== Answer ====")
print(answer.strip())


Loading weights: 100%|██████████| 311/311 [00:03<00:00, 93.10it/s] 


==== Thinking ====
<think>
Okay, so I need to figure out what 17 multiplied by 24 is. Let me think. I remember that when you multiply two numbers, you can break them down into smaller parts to make it easier. Maybe I can use the distributive property here. 

First, let me recall the distributive property: a*(b + c) = ab + ac. So, if I can split 24 into two parts that are easier to multiply with 17, that would help. Let's see... 24 can be written as 20 + 4, right? Because 20 + 4 adds up to 24. 

So then, applying the distributive property, it would be 17*(20 + 4) = 17*20 + 17*4. Let me calculate each part separately. 

First, 17 multiplied by 20. Well, 17*20... Hmm, 17*2*10. 17*2 is 34, so 34*10 is 340. Okay, so that part is 340.

Now, the second part is 17 multiplied by 4. Let me compute that. 17*4... 10*4 is 40, and 7*4 is 28, so adding those together gives 40 + 28 = 68. So, 17*4 is 68.

Now, adding those two results together: 340 + 68. Let me do that addition. 340 + 60 is 400, and th

### About the output

Everything under `Thinking` was between the `<think>` and `</think>` tags. That is the chain of thought the model wrote for itself before it answered, not the response meant for the user.

This costs tokens. The thinking is usually longer than the answer, and that is the tradeoff of a reasoning model: more compute per question for a better answer on hard problems.

If you change `enable_thinking=True` to `enable_thinking=False` and run the cell again, the `Thinking` section is empty and the model answers directly. That is how a normal instruct model behaves.
